In [ ]:
# python 3.7
# contact@leonardofilipe.com
# batch implied volatility calculation: https://github.com/jasonstrimpel

%matplotlib notebook
import scipy
import numpy             as np
import pandas            as pd
import matplotlib.pyplot as plt

from matplotlib           import cm
from scipy.stats          import norm
from scipy.optimize       import brentq
from mpl_toolkits.mplot3d import axes3d

plt.style.use('seaborn')
pd.options.mode.chained_assignment = None

def N(z):
    return norm.cdf(z)

def black_scholes_call_value(S, K, r, t, vol):
    d1 = (1.0/(vol * np.sqrt(t))) * (np.log(S/K) + (r + 0.5 * vol**2.0) * t)
    d2 = d1 - (vol * np.sqrt(t))
    return N(d1) * S - N(d2) * K * np.exp(-r * t)

def black_scholes_put_value(S, K, r, t, vol):
    d1 = (1.0/(vol * np.sqrt(t))) * (np.log(S/K) + (r + 0.5 * vol**2.0) * t)
    d2 = d1 - (vol * np.sqrt(t))
    return N(-d2) * K * np.exp(-r * t) - N(-d1) * S

def call_implied_volatility_objective_function(S, K, r, t, vol, call_option_market_price):
    return call_option_market_price - black_scholes_call_value(S, K, r, t, vol)

def call_implied_volatility(S, K, r, t, call_option_market_price, a=-2.0, b=2.0, xtol=1e-6):
    _S, _K, _r, _t, _call_option_market_price = S, K, r, t, call_option_market_price
    def fcn(vol):
        return call_implied_volatility_objective_function(_S, _K, _r, _t, vol, _call_option_market_price)
    try:
        result = brentq(fcn, a=a, b=b, xtol=xtol)
        return 0 if result <= 1.0e-6 or call_option_market_price == 0 else result
    except ValueError:
        return 0

def put_implied_volatility_objective_function(S, K, r, t, vol, put_option_market_price):
    return put_option_market_price - black_scholes_put_value(S, K, r, t, vol)

def put_implied_volatility(S, K, r, t, put_option_market_price, a=-2.0, b=2.0, xtol=1e-6):
    _S, _K, _r, _t, _put_option_market_price = S, K, r, t, put_option_market_price
    def fcn(vol):
        return put_implied_volatility_objective_function(_S, _K, _r, _t, vol, _put_option_market_price)
    try:
        result = brentq(fcn, a=a, b=b, xtol=xtol)
        return 0 if result <= 1.0e-6 or put_option_market_price == 0 else result
    except ValueError:
        return 0

def call_implied_vol_mid(series):
    S = series['Spot']
    K = series['Strike']
    r = series['Rf']
    t = series['Expiration']
    l = series['CallMid']
    return float(globals().get('call_implied_volatility')(S, K, r, t, l))

def put_implied_vol_mid(series):
    S = series['Spot']
    K = series['Strike']
    r = series['Rf']
    t = series['Expiration']
    l = series['PutMid']
    return float(globals().get('put_implied_volatility')(S, K, r, t, l))

db = pd.read_hdf("/Users/Farid Srouji/Documents/Research/master/globaldata/optiondata.h5")
db = db.drop('index',axis=1)
db['Now'] = pd.to_datetime(db['Now'])
db['Maturity'] = pd.to_datetime(db['Maturity'])
db[15:20]